In [1]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.2 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
import re

In [2]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [3]:
stop_words = set(stopwords.words("english"))
lemma = WordNetLemmatizer()
tqdm.pandas()

In [4]:
def load_and_preprocess(file_path):
    """
    Load JSON dataset, filter for at least 3 taxonomy levels,
    split into Level1–Level4, clean text, and preserve Level4 for later.
    """
    #  Load JSON file (limit rows for faster testing)
    df = pd.read_json(file_path)

    # Keep only rows with >= 2 '>' → means Level1, Level2, Level3 exist
    df = df[df['pathlist_names'].str.count('>') >= 2].reset_index(drop=True)
    df.fillna("", inplace=True)  # replace NaNs with empty strings

    #  Keep only relevant columns for processing
    cols = [
        "Title", "BrandInfo.BrandName", "ProductName", "Category.Name.Value",
        "SummaryDescription.LongSummaryDescription",
        "SummaryDescription.ShortSummaryDescription",
        "Description.LongProductName", "Description.LongDesc",
        "pathlist_names"
    ]
    df = df.reindex(columns=cols, fill_value="")  # fill missing with empty string

    #  Split taxonomy into Level1–Level4
    path_split = df['pathlist_names'].str.split('>', expand=True)
    df['Level1'] = path_split[0].str.strip()
    df['Level2'] = path_split[1].str.strip()
    df['Level3'] = path_split[2].str.strip()
    df['Level4'] = path_split[3].str.strip() if path_split.shape[1] > 3 else ""

    # Combine descriptive text fields into one column
    text_cols = [
        "Title", "BrandInfo.BrandName", "ProductName", "Category.Name.Value",
        "SummaryDescription.LongSummaryDescription",
        "SummaryDescription.ShortSummaryDescription",
        "Description.LongProductName", "Description.LongDesc"
    ]
    df["raw_text"] = df[text_cols].agg(" ".join, axis=1)

    #  Text cleaning function
    def clean_text(text):
        # keep alphanumeric + '-' and '+' (so product codes remain)
        tokens = [t.lower() for t in word_tokenize(text) if re.match(r"^[A-Za-z0-9\-\+]+$", t)]
        tokens = [lemma.lemmatize(t) for t in tokens if t not in stop_words]
        return " ".join(tokens)

    #  Apply cleaning to raw_text
    df["cleaned_text"] = df["raw_text"].progress_apply(clean_text)

    return df


In [5]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
# Process train/validation/test datasets

df_train = load_and_preprocess("icecat_data_train.json")
#df_validate = load_and_preprocess("icecat_data_validate.json")
df_test = load_and_preprocess("icecat_data_test.json")

/tmp/ipykernel_3075/4119226029.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)  # replace NaNs with empty strings
100%|██████████| 489902/489902 [10:28<00:00, 779.81it/s]
/tmp/ipykernel_3075/4119226029.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)  # replace NaNs with empty strings
100%|██████████| 153095/153095 [03:16<00:00, 779.14it/s]


In [8]:
# Save processed outputs with Level4 preserved
df_train.to_json("phase1_train_with_L4.json", orient="records", lines=True)
df_test.to_json("phase1_test_with_L4.json", orient="records", lines=True)

In [9]:
df_train.head()

,Title,BrandInfo.BrandName,ProductName,Category.Name.Value,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,Description.LongProductName,Description.LongDesc,pathlist_names,Level1,Level2,Level3,Level4,raw_text,cleaned_text
0,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,ASUS,K31CD-IT049T,PCs/Workstations,ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,"ASUS K31CD-IT049T, 3.4 GHz, 6th gen Intel® Cor...","Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,asus k31cd-it049t pc 6th gen i7 i7-6700 16 gb ...
1,HP 686915-A41 notebook spare part Keyboard,HP,686915-A41,Notebook Spare Parts,HP 686915-A41. Type: Keyboard. Keyboard langua...,"HP 686915-A41, Keyboard, Belgian, Keyboard bac...",Keyboard in midnight black finish with backlig...,,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories,Notebook Spare Parts,HP 686915-A41 notebook spare part Keyboard HP ...,hp 686915-a41 notebook spare part keyboard hp ...
2,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Fibre Optic Cables,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Get the performance you demand at a price that...,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,Computer Cables,Fibre Optic Cables,None,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,c2g 1m plenum-rated duplex single-mode fiber p...
3,HP FA889AA Battery,HP,FA889AA,Handheld Mobile Computer Spare Parts,"HP FA889AA. Product type: Battery, Product col...","HP FA889AA, Battery, White, Lithium-Ion (Li-Io...","1100 mAh, Lithium Ion, Standard Battery",Keeping an extra source of power nearby means ...,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts,None,HP FA889AA Battery HP FA889AA Handheld Mobile ...,hp fa889aa battery hp fa889aa handheld mobile ...
4,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,Lenovo,C30,PCs/Workstations,Lenovo ThinkStation C30. Processor frequency: ...,"Lenovo ThinkStation C30, 2 GHz, Intel® Xeon® E...","Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",The C30 builds on its award-winning design as ...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,lenovo thinkstation c30 e5 family e5-2620 4 gb...


In [10]:
print("✅ Phase 1 complete: Cleaned datasets with Level4 preserved")
print("Sample:\n", df_train[['pathlist_names', 'Level1', 'Level2', 'Level3', 'Level4']].head())

✅ Phase 1 complete: Cleaned datasets with Level4 preserved
Sample:
                                       pathlist_names                   Level1  \
0  Computers & Electronics>Computers>PCs/Workstat...  Computers & Electronics   
1  Computers & Electronics>Computers>Notebook Par...  Computers & Electronics   
2  Computers & Electronics>Computer Cables>Fibre ...  Computers & Electronics   
3  Computers & Electronics>Computers>Handheld Mob...  Computers & Electronics   
4  Computers & Electronics>Computers>PCs/Workstat...  Computers & Electronics   

            Level2                                Level3                Level4  
0        Computers                      PCs/Workstations                  None  
1        Computers          Notebook Parts & Accessories  Notebook Spare Parts  
2  Computer Cables                    Fibre Optic Cables                  None  
3        Computers  Handheld Mobile Computer Spare Parts                  None  
4        Computers                      